# AUXNEW C8 V2 — EXPERIMENTAL, DISABLED

Un seul candidat : supervision auxiliaire BCE .25, tête primaire seule en inférence. Huit canaux C8 inchangés, quatre époques, seed42, seuil .78. Aucun fit automatique. Autorisation utilisateur distincte requise avant toute activation. Onze partitions existantes, Internet OFF. Test 2021 fermé. Aucun appel FIRE13.


In [ ]:
RUN_GPU_PREFLIGHT = False
RUN_FIT = False
import os,sys
if 'torch' in sys.modules: raise RuntimeError('Restart kernel first')
os.environ['CUDA_VISIBLE_DEVICES']='0'


In [ ]:
from pathlib import Path
import base64,hashlib,io,zipfile,json
DATA=base64.b64decode('...')
assert hashlib.sha256(DATA).hexdigest()=='dc56411cbe086747e866fc4c4da8dc7097e3d4a2e28d9d34abf51ff62f27a8fc'
ROOT=Path('/kaggle/working/geofirex-auxnew-code')
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(DATA)) as z:
    for n in z.namelist():assert (ROOT/n).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
sys.path.insert(0,str(ROOT/'src'))
from geofirex import auxnew_experiment as c
print(c.identity(ROOT)[2])


In [ ]:
EXPECTED={r['map_sha256'] for r in c.old.read(ROOT/c.old.REPORT/'partitions.json').values()}
MAPS=[p for p in Path('/kaggle/input').rglob('mount-map.json') if c.sha256(p) in EXPECTED]
assert len(MAPS)==11
print(c.old.preflight(ROOT,MAPS))


Ne pas créer de grant sans nouvelle autorisation utilisateur explicite. Copier les trois templates DISABLED dans /kaggle/working, renseigner opérateur/date/identifiant unique/quota actuel/deux sauvegardes. Réserver 300 minutes, 10 heures-carte, 16/8 GiB RAM totale/disponible et 20 GiB disque. Le temps CPU ne qualifie pas le GPU. Contrôler T4, versions exactes et cuDNN ; aucun pip automatique. Ne pas supprimer les verrous.


In [ ]:
GPU_AUTH=Path('/kaggle/working/auxnew_gpu_authorization.json')
FIT_AUTH=Path('/kaggle/working/auxnew_fit_authorization.json')
EVAL_AUTH=Path('/kaggle/working/auxnew_evaluate_authorization.json')
OUT=Path('/kaggle/working')/c.RUN
if RUN_GPU_PREFLIGHT:
    c.authorize(ROOT,GPU_AUTH,'auxnew-gpu-preflight',OUT.parent)
    print(c.preflight(ROOT,MAPS,OUT.parent,gpu=True))
else: print('GPU disabled')


In [ ]:
if RUN_FIT:
    assert RUN_GPU_PREFLIGHT
    c.fit(ROOT,MAPS,OUT,FIT_AUTH,EVAL_AUTH,GPU_AUTH)
else: print('Fit disabled; no optimizer created')


In [ ]:
from IPython.display import display,FileLink
for p in sorted(Path('/kaggle/working').glob(c.RUN+'-*.zip')):
    print(c.old.verify_zip(p,p.with_suffix('.sha256').read_text().split()[0]))
    display(FileLink(str(p)));display(FileLink(str(p.with_suffix('.sha256'))))
